# Working with complicated dataset

Your name:

### Question 1

blast_results: https://raw.githubusercontent.com/csbfx/advpy122-data/master/blast_results.csv

Read in the data from the csv file above. Skip all the comment lines, create a header for the dataframe based on the fields list in the comment line that starts with **# Fields:**. Drop the first column `query acc.ver`. Set the `subject acc.ver` as the index of the dataframe. The last column of the dataframe should be `publications`. Show the first five rows of the resulting dataframe.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from urllib.request import urlopen
from io import StringIO

url = 'https://raw.githubusercontent.com/csbfx/advpy122-data/master/blast_results.csv'
response = urlopen(url)
content = response.read().decode('utf-8')
lines = content.strip().split('\n')

# Extract header from the "# Fields:" line
header = None
data_lines = []
for line in lines:
    if line.startswith('# Fields:') or line.startswith('"# Fields:'):
        # Extract field names - remove the "# Fields: " prefix and any quotes
        fields_part = line.replace('"# Fields:', '').replace('# Fields:', '')
        # The fields are comma-separated, only take the actual field names (not empty trailing parts)
        header = [col.strip().strip('"') for col in fields_part.split(',') if col.strip() and not col.strip() == '']
        header = header[:15]  # There are exactly 15 fields
    elif not line.startswith('#') and not line.startswith('"#'):
        data_lines.append(line)

# Read the data - pandas will handle the quoted fields with commas correctly
data_text = '\n'.join(data_lines)
df = pd.read_csv(StringIO(data_text), header=None, usecols=range(15), names=header)

# Drop the first column 'query acc.ver'
df = df.drop(columns=['query acc.ver'])

# Set 'subject acc.ver' as the index
df = df.set_index('subject acc.ver')

df.head()

KeyError: "['query acc.ver'] not found in axis"

### Question 2
What is the average number of publications?

In [ ]:
publications = pd.to_numeric(df['publications'], errors='coerce')
avg_publications = publications.mean()
print(f"Average number of publications: {avg_publications:.2f}")

### Question 3
List the `subject acc.ver` that has over 15,000 bonds.

In [ ]:
bonds = df['bonds'].str.replace(',', '').str.strip().astype(int)
over_15000 = df[bonds > 15000].index.tolist()
print(f"Subject acc.ver with over 15,000 bonds:")
for acc in over_15000:
    print(acc)

### Question 4
Create a plot that shows the correlation between `% identity` and `% positives`.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['% identity'], df['% positives'], alpha=0.6, edgecolors='black', linewidth=0.5)
plt.xlabel('% Identity')
plt.ylabel('% Positives')
plt.title('Correlation between % Identity and % Positives')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Question  5
Add a column `Protein source` to the dataframe from Question 1 based on the data in this csv file: https://raw.githubusercontent.com/csbfx/advpy122-data/master/protein_source.csv. Merge the data when the `subject acc.ver` from the dataframe equals to `Protein` in the csv file. Entries without a matching protein will have `NA` as `Protein source`.

In [ ]:
protein_url = 'https://raw.githubusercontent.com/csbfx/advpy122-data/master/protein_source.csv'
protein_df = pd.read_csv(protein_url, index_col=0)

df_merged = df.reset_index()
df_merged = df_merged.merge(
    protein_df[['Protein', 'Source']], 
    left_on='subject acc.ver', 
    right_on='Protein', 
    how='left'
)

df_merged = df_merged.rename(columns={'Source': 'Protein source'})
df_merged = df_merged.drop(columns=['Protein'])
df_merged = df_merged.set_index('subject acc.ver')
df_merged